# 28 · GraphSAGE on the text-similarity graph (E65)

Graph: 10 nearest neighbours by e5 embedding within the platform. Node features: final-model inputs + embedding PCA 16. GraphSAGE vs the same network without the graph vs CatBoost; 5-fold brand CV.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.width", 220, "display.max_columns", 30)
import extra_models as em, history as H, context_text as ct, gnn

In [2]:
f, meta = em.load(ROOT); E = ct.embeddings(ROOT, ROOT / "data" / "processed" / "embeddings_e5_cache.pkl"); pe, _ = ct.reduce(E, ROOT); print(gnn.CFG)

{'hidden': 64, 'dropout': 0.2, 'lr': 0.003, 'weight_decay': 0.0001, 'epochs': 400, 'k': 10, 'seeds': (0, 1, 2)}


In [3]:
P = gnn.run(f, meta, E, pe); S = H.summarise(P)
S.pivot(index="model", columns="platform", values="MAE")[gnn.PLATS].round(3)

platform,tiktok,youtube,twitter
model,,,
CatBoost (final model),0.273,0.456,0.466
GraphSAGE (text graph),0.279,0.467,0.474
"MLP (same features, no graph)",0.274,0.456,0.467
mean CatBoost + GraphSAGE,0.266,0.440,0.451
mean CatBoost + MLP (control),0.265,0.440,0.451


In [4]:
G = pd.DataFrame([{"platform": p, "base": a, "model": b, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(P, p, a, b)))} for p in gnn.PLATS for a, b in [("MLP (same features, no graph)", "GraphSAGE (text graph)"), ("CatBoost (final model)", "GraphSAGE (text graph)"), ("CatBoost (final model)", "mean CatBoost + GraphSAGE"), ("CatBoost (final model)", "mean CatBoost + MLP (control)"), ("mean CatBoost + MLP (control)", "mean CatBoost + GraphSAGE")]]); G.round(3)

,platform,base,model,gain,CI low,CI high
0,tiktok,"MLP (same features, no graph)",GraphSAGE (text graph),-0.006,-0.011,-0.001
1,tiktok,CatBoost (final model),GraphSAGE (text graph),-0.006,-0.012,0.001
2,tiktok,CatBoost (final model),mean CatBoost + GraphSAGE,0.007,0.004,0.011
3,tiktok,CatBoost (final model),mean CatBoost + MLP (control),0.008,0.005,0.011
4,tiktok,mean CatBoost + MLP (control),mean CatBoost + GraphSAGE,-0.001,-0.003,0.002
5,youtube,"MLP (same features, no graph)",GraphSAGE (text graph),-0.011,-0.020,-0.002
6,youtube,CatBoost (final model),GraphSAGE (text graph),-0.011,-0.024,0.000
7,youtube,CatBoost (final model),mean CatBoost + GraphSAGE,0.015,0.008,0.022
8,youtube,CatBoost (final model),mean CatBoost + MLP (control),0.016,0.009,0.021
9,youtube,mean CatBoost + MLP (control),mean CatBoost + GraphSAGE,-0.000,-0.005,0.005


## Takeaways
See `docs/en/experiments.md` (E65). Numbers: `data/processed/gnn_numbers.json`.

In [5]:
json.dump({"config": {k: v for k, v in gnn.CFG.items()}, "models": S.to_dict("records"), "gains": G.to_dict("records")}, open(ROOT / "data" / "processed" / "gnn_numbers.json", "w"), indent=1, default=float)